# Dunnhumby seed 48: 데이터 적응형 N/V 표현 M2

- 기존 seed 48의 M1·고정계수 M2 결과를 신원 검증 후 재사용합니다.
- 새 M2 **1개만** 300 epoch 학습합니다.
- N·V·아이템 경제속성의 혼합계수와 총 개입강도가 학습 데이터에서 초기화되고 동일 BPR로 공동 학습됩니다.
- Dunnhumby development 684–690만 사용하며 마지막 주 test·holdout은 열지 않습니다.
- 고정 300 epoch 마지막 체크포인트를 비교하며 조기종료·epoch 선택은 없습니다.

In [ ]:
from pathlib import Path
from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir(), 'Google Drive가 연결되지 않았습니다.'

In [ ]:
import subprocess, sys, torch

SOURCE_COMMIT = 'a17035f97c23'
REPO = Path('/content/clv-m2-adaptive-' + SOURCE_COMMIT)
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', SOURCE_COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short=12', 'HEAD'], text=True).strip()
assert actual == SOURCE_COMMIT
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
sys.path.insert(0, str(REPO))
print('source:', SOURCE_COMMIT, '| GPU:', torch.cuda.get_device_name(0))

In [ ]:
import clv_m2_dataset_adaptive_orthogonal_nv_seed48 as screen

cfg, prepared = screen.prepare()
# 여기까지는 학습 없음. 초기 N/V/가격 혼합계수와 rho를 출력합니다.
prepared['feature_settings']['dataset_initialization']

In [ ]:
result = screen.run(cfg, prepared)
print(result['reading'])

In [ ]:
from IPython.display import display

metrics = ['model_id', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
           'recall@50', 'ndcg@50', 'price_purchase_amount_weighted_hit@10',
           'vndcg@10', 'price_purchase_amount_weighted_hit@50', 'vndcg@50']
display(result['absolute'].loc[result['absolute']['epoch'].eq(300), metrics])
display(result['comparison'].loc[result['comparison']['epoch'].eq(300)].sort_values(['reference', 'metric']))
adaptive = result['absolute'].loc[result['absolute']['model_id'].eq(screen.M2_ADAPTIVE)]
parameter_columns = ['epoch', 'axis_weight_n', 'axis_weight_v', 'axis_weight_price', 'adaptive_rho']
display(adaptive[[column for column in parameter_columns if column in adaptive.columns]])

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

output = Path('/content/clv_m2_dataset_adaptive_orthogonal_nv_seed48_results.zip')
with ZipFile(output, 'w', ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, Path(path).name)
    diagnostic = Path(cfg.out_dir) / 'feature_diagnostic.json'
    archive.write(diagnostic, diagnostic.name)
files.download(str(output))